In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

file_path = '../data/raw/Data_Professional_Salary_Survey_Responses.xlsx'

df = pd.read_excel(file_path, header=3)
df = df.loc[:, ~df.columns.astype(str).str.contains('^Unnamed')]
df.columns = df.columns.astype(str).str.strip()

print(f"Data loaded successfully. Initial shape: {df.shape}")

Data loaded successfully. Initial shape: (14869, 22)


In [2]:
# Drop columns that do not provide predictive value
columns_to_drop = ['Survey Year', 'Timestamp', 'PostalCode']
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df = df.drop(columns=columns_to_drop)
print(f"Shape after feature selection: {df.shape}")

Shape after feature selection: (14869, 19)


In [3]:
# Feature Engineering
def categorize_experience(years):
    if pd.isna(years): 
        return 'Not Specified'
    if years <= 3: 
        return 'Junior'
    elif years <= 8: 
        return 'Mid'
    else: 
        return 'Senior'

if 'YearsWithThisTypeOfJob' in df.columns:
    df['Experience_Level'] = df['YearsWithThisTypeOfJob'].apply(categorize_experience)

if 'YearsWithThisDatabase' in df.columns and 'YearsWithThisTypeOfJob' in df.columns:
    df['Specialization_Ratio'] = df['YearsWithThisDatabase'] / (df['YearsWithThisTypeOfJob'] + 0.1)

if 'YearsWithThisTypeOfJob' in df.columns:
    df['Years_Squared'] = df['YearsWithThisTypeOfJob'] ** 2

print(f"Features created successfully. Current df shape: {df.shape}")

Features created successfully. Current df shape: (14869, 22)


In [4]:
# Dynamic identification of column types
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Handle missing values
for col in categorical_cols:
    df[col] = df[col].fillna('Not Specified')

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"Missing values handled. Max nulls remaining: {df.isnull().sum().max()}")

Missing values handled. Max nulls remaining: 0


In [5]:
# Outliers handling and target transformation
lower_bound = 5000
upper_bound = 500000

df = df[(df['SalaryUSD'] >= lower_bound) & (df['SalaryUSD'] <= upper_bound)]
df['SalaryUSD_Log'] = np.log1p(df['SalaryUSD'])

print(f"Shape after handling outliers: {df.shape}")

Shape after handling outliers: (14755, 23)


In [6]:
# Categorical Encoding
def group_rare_categories(series, threshold=0.02):
    value_counts = series.value_counts(normalize=True)
    rare_categories = value_counts[value_counts < threshold].index
    return series.replace(rare_categories, 'Other')

for col in ['JobTitle', 'Country', 'PrimaryDatabase']:
    if col in df.columns:
        df[col] = group_rare_categories(df[col])

career_col = [col for col in df.columns if 'CareerPlans' in col][0]
features_to_encode = [col for col in categorical_cols if col != career_col and col in df.columns]

df_encoded = pd.get_dummies(df, columns=features_to_encode, drop_first=True)
print(f"Shape after One-Hot Encoding: {df_encoded.shape}")

Shape after One-Hot Encoding: (14755, 2952)


In [7]:
# Data Scaling
targets = ['SalaryUSD', 'SalaryUSD_Log', career_col]
num_features_to_scale = [col for col in numerical_cols if col not in targets and col in df_encoded.columns]

scaler = StandardScaler()

if num_features_to_scale:
    df_encoded[num_features_to_scale] = scaler.fit_transform(df_encoded[num_features_to_scale])

print("Scaling completed.")

Scaling completed.


In [8]:
# Feature Engineering Verification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

y_verify = df['SalaryUSD_Log']

# Dynamically select columns to avoid data type mismatch errors
base_categorical = [c for c in categorical_cols if c != career_col and c != 'Experience_Level']
base_numerical = [c for c in numerical_cols if c not in ['SalaryUSD', 'SalaryUSD_Log', 'Specialization_Ratio', 'Years_Squared']]

# Prepare baseline data
df_baseline = df[base_categorical + base_numerical].copy()
X_baseline = pd.get_dummies(df_baseline, columns=base_categorical, drop_first=True)
X_baseline[base_numerical] = scaler.fit_transform(X_baseline[base_numerical])

# Prepare enhanced data
enhanced_categorical = base_categorical + ['Experience_Level']
enhanced_numerical = base_numerical + ['Specialization_Ratio', 'Years_Squared']

df_enhanced = df[enhanced_categorical + enhanced_numerical].copy()
X_enhanced = pd.get_dummies(df_enhanced, columns=enhanced_categorical, drop_first=True)
X_enhanced[enhanced_numerical] = scaler.fit_transform(X_enhanced[enhanced_numerical])

# Train and evaluate models
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_baseline, y_verify, test_size=0.2, random_state=42)
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_enhanced, y_verify, test_size=0.2, random_state=42)

model_b = Ridge(alpha=10.0)
model_b.fit(X_train_b, y_train_b)
preds_b = model_b.predict(X_test_b)
mae_b = mean_absolute_error(np.expm1(y_test_b), np.expm1(preds_b))
r2_b = r2_score(y_test_b, preds_b)

model_e = Ridge(alpha=10.0)
model_e.fit(X_train_e, y_train_e)
preds_e = model_e.predict(X_test_e)
mae_e = mean_absolute_error(np.expm1(y_test_e), np.expm1(preds_e))
r2_e = r2_score(y_test_e, preds_e)

print(f"Baseline MAE: ${mae_b:,.2f} | Baseline R2: {r2_b:.4f}")
print(f"Enhanced MAE: ${mae_e:,.2f} | Enhanced R2: {r2_e:.4f}")

Baseline MAE: $23,346.15 | Baseline R2: 0.4677
Enhanced MAE: $22,903.44 | Enhanced R2: 0.4873


### NOTE ON LINEAR MODEL LIMITATIONS & FEATURE ENGINEERING VERIFICATION

**Why this cell is preserved:**
This verification step intentionally illustrates a key machine learning phenomenon regarding high-cardinality categorical data and linear models:

1. **Initial Proof of Concept (Clean State):** During early iterations with controlled encoding, the engineered features proved their predictive power, improving the baseline score:
   * *Baseline:* MAE = $23,346.15 | R² = 0.4677
   * *Enhanced:* MAE = $22,903.44 | R² = 0.4873 (Features reduced error and captured variance better).

2. **The High-Cardinality Crash (Current State):**
   Once full categorical data (such as specific job titles and countries) was unpacked via One-Hot Encoding, the feature space expanded to over 2,900 columns. As a result, standard linear estimators (Linear Regression / Ridge) suffer from extreme overfitting, leading to an inflation of MAE (~$33k-$34k) and a negative R² score (~ -2.1).

**Conclusion:**
The enhanced model still outperforms the baseline (lower MAE), confirming the new features are fundamentally useful. However, this catastrophic drop in R² serves as a strict architectural justification for transitioning to non-linear, tree-based ensembles (Random Forest, XGBoost) in the next phase (`03_modeling_regression.ipynb`), which natively handle high-dimensional sparse matrices.

In [9]:
#DO NOT DELETE
# Feature 1: Experience Level Discretization
def categorize_experience(years):
    if years <= 3:
        return 'Junior'
    elif years <= 8:
        return 'Mid'
    else:
        return 'Senior'

if 'YearsWithThisTypeOfJob' in df.columns:
    df['Experience_Level'] = df['YearsWithThisTypeOfJob'].apply(categorize_experience)
    if 'Experience_Level' not in categorical_cols:
        categorical_cols.append('Experience_Level')

# Feature 2: Specialization Ratio Interaction
if 'YearsWithThisDatabase' in df.columns and 'YearsWithThisTypeOfJob' in df.columns:
    df['Specialization_Ratio'] = df['YearsWithThisDatabase'] / (df['YearsWithThisTypeOfJob'] + 0.1)
    if 'Specialization_Ratio' not in numerical_cols:
        numerical_cols.append('Specialization_Ratio')

# Feature 3: Polynomial Transformation (Years Squared)
if 'YearsWithThisTypeOfJob' in df.columns:
    df['Years_Squared'] = df['YearsWithThisTypeOfJob'] ** 2
    if 'Years_Squared' not in numerical_cols:
        numerical_cols.append('Years_Squared')



# verification

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

y_verify = df_encoded['SalaryUSD_Log']
career_col = [col for col in df_encoded.columns if 'CareerPlans' in col][0]

X_all_features = df_encoded.drop(columns=['SalaryUSD', 'SalaryUSD_Log', career_col])

new_features = ['Specialization_Ratio', 'Years_Squared'] + [col for col in df_encoded.columns if 'Experience_Level' in col]
X_baseline_features = X_all_features.drop(columns=new_features, errors='ignore')

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_all_features, y_verify, test_size=0.2, random_state=42)
X_train_b, X_test_b, _, _ = train_test_split(X_baseline_features, y_verify, test_size=0.2, random_state=42)

# Baseline Model (Without new features)
model_baseline = LinearRegression()
model_baseline.fit(X_train_b, y_train_a)
preds_baseline = model_baseline.predict(X_test_b)
mae_baseline = mean_absolute_error(np.expm1(y_test_a), np.expm1(preds_baseline))
r2_baseline = r2_score(y_test_a, preds_baseline)

# Enhanced Model (With new features)
model_enhanced = LinearRegression()
model_enhanced.fit(X_train_a, y_train_a)
preds_enhanced = model_enhanced.predict(X_test_a)
mae_enhanced = mean_absolute_error(np.expm1(y_test_a), np.expm1(preds_enhanced))
r2_enhanced = r2_score(y_test_a, preds_enhanced)

print(f"Baseline MAE: ${mae_baseline:,.2f} | Baseline R2: {r2_baseline:.4f}")
print(f"Enhanced MAE: ${mae_enhanced:,.2f} | Enhanced R2: {r2_enhanced:.4f}")

Baseline MAE: $34,153.68 | Baseline R2: -2.3345
Enhanced MAE: $33,439.08 | Enhanced R2: -2.1067


In [10]:
# Save processed data
output_path = '../data/processed/processed_data.csv'
os.makedirs('../data/processed', exist_ok=True)

df_encoded.to_csv(output_path, index=False)
print(f"Processed dataset saved to: {output_path}")

Processed dataset saved to: ../data/processed/processed_data.csv
